# MAR20 Aircraft Detection - UNet Training on Google Colab

Self-contained notebook for training the MAR20 UNet on a Colab GPU.

This single notebook contains everything that lives in the local repo
(`model/unet_layers.py`, `model/unet_model.py`, `provider/dataset_provider.py`,
`training.py`, `result_script.py`), so no other files need to be uploaded.

- `torch` / `torchvision` are **already installed on Colab** (CUDA build) and are
  deliberately *not* reinstalled here.
- The dataset is loaded from an uploaded `MAR20.zip` (~1.2 GB) or straight from
  Google Drive.
- Checkpoints (one per epoch + `best_model.pt`), the loss curve and prediction
  images are written to the local VM and can be copied to Drive / downloaded at
  the end (the runtime is erased once the session closes).

### How to use
1. `Runtime -> Change runtime type` and pick a **GPU** accelerator (T4 / A100).
2. Run the cells top to bottom. The dataset cell either uploads `MAR20.zip`
   or reads it from Drive if you set `DRIVE_ZIP` / `DRIVE_FOLDER`.
3. In the *Training* cell adjust `NUM_EPOCHS` to fit your time budget. With your
   CPU-only setup one epoch takes ~1 h 20 min; on a T4 GPU an epoch typically
   takes only a few minutes, so the full 20-epoch run becomes feasible. Setting
   `USE_AMP = True` gives roughly another 1.5-3x speed-up on GPU.

In [ ]:
import torch
print("torch", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU memory (GB):", torch.cuda.get_device_properties(0).total_memory / 1e9)
if not torch.cuda.is_available():
    print("WARNING: no GPU detected - training will be very slow (you reported ~80 min/epoch).")
    print("Use Runtime > Change runtime type > T4 GPU.")

In [ ]:
# Everything this notebook imports (torch, torchvision, PIL, numpy,
# matplotlib, tqdm) is preinstalled on Colab, so nothing extra is required.
# Uncomment the line below only if you hit a missing-module error:
# !pip install -q albumentations scikit-image
print("No additional installs required.")

## 1. Provide the dataset

The dataset (Pascal-VOC style) looks like this:

```
data/MAR20/
  JPEGImages/           # 3842 .jpg images
  Annotations/Horizontal Bounding Boxes/  # .xml files with bndbox annotations
  ImageSets/Main/       # train.txt (1132), val.txt (199), test.txt (2510)
```

Choose one:
- **Upload**: leave `DRIVE_ZIP` / `DRIVE_FOLDER` empty and the cell will show an
  upload widget (paste `MAR20.zip`).
- **Google Drive**: mount Drive and set `DRIVE_ZIP` (path to the `.zip`) or
  `DRIVE_FOLDER` (path to an already-extracted `MAR20` folder).

In [ ]:
# Optional - only needed if you want to read the data from Google Drive.
from google.colab import drive
drive.mount("/content/drive")
print("Drive mounted. Set DRIVE_ZIP or DRIVE_FOLDER in the next cell.")

In [ ]:
import os
import shutil
import zipfile

DATA_DIR = "/content/mar20_data"
DATA_ROOT = os.path.join(DATA_DIR, "MAR20")   # expected dataset directory

# --- Set ONE of the two Drive sources below, or leave both empty to upload ---
DRIVE_ZIP = ""            # e.g. "/content/drive/MyDrive/dl4rs_MAR20/data/MAR20.zip"
DRIVE_FOLDER = ""         # e.g. "/content/drive/MyDrive/dl4rs_MAR20/data/MAR20"

os.makedirs(DATA_DIR, exist_ok=True)

def dataset_ready():
    return os.path.isfile(os.path.join(DATA_ROOT, "ImageSets", "Main", "train.txt"))

def extract_zip(zip_path):
    print(f"Unzipping {zip_path} -> {DATA_DIR} (takes a few minutes)...")
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(DATA_DIR)

if dataset_ready():
    print("Dataset already extracted:", DATA_ROOT)
elif DRIVE_FOLDER and os.path.isdir(DRIVE_FOLDER):
    print("Copying extracted dataset from Drive ...")
    shutil.copytree(DRIVE_FOLDER, DATA_ROOT)
elif DRIVE_ZIP and os.path.isfile(DRIVE_ZIP):
    extract_zip(DRIVE_ZIP)
else:
    from google.colab import files
    print("Upload MAR20.zip (~1.2 GB) with the widget below ...")
    uploaded = files.upload()
    zip_path = next(iter(uploaded))
    extract_zip(zip_path)

assert dataset_ready(), "Dataset not found - check DRIVE_ZIP/DRIVE_FOLDER or re-upload."
print("Dataset ready:", DATA_ROOT)

## 2. Model definition

`model/unet_layers.py` -> `DoubleConv` (two conv + BN + ReLU blocks).

In [ ]:
import torch
import torch.nn as nn


class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(DoubleConv, self).__init__()
        self.dc = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=(3, 3), padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(),
            nn.Conv2d(out_channels, out_channels, kernel_size=(3, 3), padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU()
        )

    def forward(self, x):
        return self.dc(x)

In [ ]:
# `model/unet_model.py` -> UNetModel (5 encoder + 4 decoder stages).
class UNetModel(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(UNetModel, self).__init__()

        # Encoder part
        self.dc1 = DoubleConv(in_channels, 64)
        self.dc2 = DoubleConv(64, 128)
        self.dc3 = DoubleConv(128, 256)
        self.dc4 = DoubleConv(256, 512)
        self.dc5 = DoubleConv(512, 1024)
        self.max_pool = nn.MaxPool2d(2)

        # Decoder part
        self.tc4 = nn.ConvTranspose2d(1024, 512, kernel_size=2, stride=2)
        self.tc3 = nn.ConvTranspose2d(512, 256, kernel_size=2, stride=2)
        self.tc2 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.tc1 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)

        self.dc6 = DoubleConv(1024, 512)
        self.dc7 = DoubleConv(512, 256)
        self.dc8 = DoubleConv(256, 128)
        self.dc9 = DoubleConv(128, 64)

        self.final_conv = nn.Conv2d(64, out_channels, kernel_size=1)

    def forward(self, x):
        skip_connections = []

        # Encoder passthrough / forward-pass
        x = self.dc1(x)
        skip_connections.append(x)
        x = self.max_pool(x)

        x = self.dc2(x)
        skip_connections.append(x)
        x = self.max_pool(x)

        x = self.dc3(x)
        skip_connections.append(x)
        x = self.max_pool(x)

        x = self.dc4(x)
        skip_connections.append(x)
        x = self.max_pool(x)

        x = self.dc5(x)

        # Decoder
        x = self.tc4(x)
        x = torch.cat((x, skip_connections[3]), dim=1)
        x = self.dc6(x)

        x = self.tc3(x)
        x = torch.cat((x, skip_connections[2]), dim=1)
        x = self.dc7(x)

        x = self.tc2(x)
        x = torch.cat((x, skip_connections[1]), dim=1)
        x = self.dc8(x)

        x = self.tc1(x)
        x = torch.cat((x, skip_connections[0]), dim=1)
        x = self.dc9(x)

        return self.final_conv(x)

## 3. Dataset loader

`provider/dataset_provider.py`: loads images, rasterizes all bounding boxes
from the XML annotations into one binary mask, resizes both to 512x512.
`get_loader()` builds train / val / test DataLoaders. On Colab the whole split
is preloaded into RAM (fast, ~4.6 GB for the 1132-image train split).

In [ ]:
import os
import xml.etree.ElementTree as ET

import numpy as np
import torch
import torch.utils.data as td
from PIL import Image
from tqdm import tqdm
from torchvision.transforms import Resize

TARGET_SIZE = (512, 512)


class MAR20Dataset(td.Dataset):
    """Each sample is an (image, mask) pair with shape (3,512,512) / (1,512,512)."""

    def __init__(self, dataset_root, split="train", transform=None):
        self.dataset_root = dataset_root
        self.transform = transform

        split_file = os.path.join(dataset_root, "ImageSets", "Main", f"{split}.txt")
        with open(split_file, "r") as f:
            self.image_ids = [line.strip() for line in f if line.strip()]

        # Preload all images and masks to avoid disk I/O during training.
        self.samples = []
        for img_id in tqdm(self.image_ids, desc=f"Loading {split} set"):
            image = self._load_image(img_id)
            mask = self._load_mask(img_id)
            self.samples.append((image, mask))

    def _load_image(self, img_id):
        img_path = os.path.join(self.dataset_root, "JPEGImages", f"{img_id}.jpg")
        pil_image = Image.open(img_path).convert("RGB")
        np_image = np.array(pil_image)
        tensor = torch.from_numpy(np_image).permute(2, 0, 1).float() / 255.0
        tensor = Resize(TARGET_SIZE)(tensor)
        return tensor

    def _load_mask(self, img_id):
        xml_path = os.path.join(
            self.dataset_root, "Annotations", "Horizontal Bounding Boxes", f"{img_id}.xml"
        )
        tree = ET.parse(xml_path)
        root = tree.getroot()

        width = int(root.find("size/width").text)
        height = int(root.find("size/height").text)
        mask = np.zeros((height, width), dtype=np.uint8)

        for obj in root.findall("object"):
            bndbox = obj.find("bndbox")
            xmin = int(float(bndbox.find("xmin").text))
            ymin = int(float(bndbox.find("ymin").text))
            xmax = int(float(bndbox.find("xmax").text))
            ymax = int(float(bndbox.find("ymax").text))
            mask[ymin:ymax, xmin:xmax] = 1

        mask_tensor = torch.from_numpy(mask).unsqueeze(0).float()
        mask_tensor = Resize(TARGET_SIZE, interpolation=Image.NEAREST)(mask_tensor)
        return mask_tensor

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        image, mask = self.samples[index]
        if self.transform is not None:
            image = self.transform(image)
        return image, mask


def get_loader(dataset_type, batch_size, shuffle=True):
    """Build a DataLoader for the MAR20 dataset (DATA_ROOT is defined above)."""
    dataset = MAR20Dataset(dataset_root=DATA_ROOT, split=dataset_type)
    return td.DataLoader(dataset=dataset, batch_size=batch_size, shuffle=shuffle)

## 4. Training helpers

`training.py`: one train step and one validation step per epoch. If `USE_AMP`
is enabled, forward passes run in mixed precision (float16) via `autocast`
+ `GradScaler`.

In [ ]:
from tqdm import tqdm


def train(model, loss_fn, optimizer, epoch, train_ds, device, scaler=None):
    """Run one training epoch. Returns the average loss over all batches."""
    model.train()
    running_loss = []
    loop = tqdm(train_ds, desc=f"Train epoch {epoch}")

    for (data, target) in loop:
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad(set_to_none=True)

        with torch.autocast(device_type="cuda", dtype=torch.float16,
                            enabled=scaler is not None):
            prediction = model(data)
            loss = loss_fn(prediction, target)

        if scaler is not None:
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            optimizer.step()

        running_loss.append(loss.item())
        loop.set_postfix(loss=f"{loss.item():.4f}")

    return np.mean(running_loss)


def validation(model, loss_fn, epoch, valid_ds, device):
    """Run one validation epoch. Returns the average loss over all batches."""
    model.eval()
    running_loss = []
    loop = tqdm(valid_ds, desc=f"Val   epoch {epoch}")

    for (data, target) in loop:
        data, target = data.to(device), target.to(device)
        with torch.no_grad():
            with torch.autocast(device_type="cuda", dtype=torch.float16,
                                enabled=device.type == "cuda"):
                prediction = model(data)
                loss = loss_fn(prediction, target)

        running_loss.append(loss.item())
        loop.set_postfix(loss=f"{loss.item():.4f}")

    return np.mean(running_loss)

## 5. Run the training

Checkpoints are saved to `/content/mar20_checkpoints`:

- `model_epoch<N>.pt` - after every epoch
- `best_model.pt`     - epoch with the lowest validation loss (re-saved each time it improves)
- `loss_curve.png`    - train/val loss over time

In [ ]:
import os
import time

import matplotlib
matplotlib.use("Agg")  # non-interactive - saves plots to file
import matplotlib.pyplot as plt
import numpy as np
import torch.nn as nn
import torch.optim as opt

# ---------- Hyperparameters ----------
NUM_EPOCHS       = 20         # reduce if you are on a tight time budget
LEARNING_RATE    = 0.001
TRAIN_BATCH_SIZE = 4
VAL_BATCH_SIZE   = 4
SEED             = 42
USE_AMP          = False      # True = float16 mixed precision (~1.5-3x faster on GPU)

torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Training device:", device)

model = UNetModel(in_channels=3, out_channels=1).to(device)
loss_fn = nn.BCEWithLogitsLoss()
optimizer = opt.Adam(model.parameters(), lr=LEARNING_RATE)

train_ds = get_loader(dataset_type="train", batch_size=TRAIN_BATCH_SIZE, shuffle=True)
val_ds   = get_loader(dataset_type="val",   batch_size=VAL_BATCH_SIZE,   shuffle=False)

scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP and device.type == "cuda")

CKPT_DIR = "/content/mar20_checkpoints"
os.makedirs(CKPT_DIR, exist_ok=True)

all_tr_losses, all_val_losses = [], []
best_val_loss = float("inf")
epoch_times = []

for epoch in range(NUM_EPOCHS):
    t0 = time.time()
    tr_loss = train(model, loss_fn, optimizer, epoch, train_ds, device, scaler)
    val_loss = validation(model, loss_fn, epoch, val_ds, device)

    all_tr_losses.append(tr_loss)
    all_val_losses.append(val_loss)
    epoch_times.append(time.time() - t0)

    print(f"Epoch {epoch}: train_loss={tr_loss:.4f}, val_loss={val_loss:.4f} "
          f"[{epoch_times[-1]/60:.1f} min/epoch]")

    checkpoint = {
        "epoch": epoch,
        "net_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "train_loss": tr_loss,
        "val_loss": val_loss,
    }
    torch.save(checkpoint, os.path.join(CKPT_DIR, f"model_epoch{epoch}.pt"))

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(checkpoint, os.path.join(CKPT_DIR, "best_model.pt"))

avg_min = np.mean(epoch_times) / 60.0
print(f"Average per epoch: {avg_min:.1f} min")
print(f"Local CPU setup was ~80 min/epoch - GPU is {80/avg_min:.0f}x faster here.")
print(f"ETA for all {NUM_EPOCHS} epochs: {avg_min * NUM_EPOCHS / 60:.1f} h")

# Loss curve
plt.figure()
plt.plot(all_tr_losses, color="blue", label="Train")
plt.plot(all_val_losses, color="red", label="Validation")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.title("Training and Validation Loss")
plt.savefig(os.path.join(CKPT_DIR, "loss_curve.png"), dpi=150)
print(f"Loss curve  : {CKPT_DIR}/loss_curve.png")
print(f"Best model  : {CKPT_DIR}/best_model.pt  (val_loss={best_val_loss:.4f})")

## 6. (Optional) Visualize predictions on the test set

Loads `best_model.pt`, runs inference on the `test` split and saves
prediction vs. ground-truth pairs (raw logits are passed through `sigmoid`).

In [ ]:
import os
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from tqdm import tqdm


def visualize_results(num_samples=100):
    ckpt = os.path.join(CKPT_DIR, "best_model.pt")
    if not os.path.exists(ckpt):
        print("No checkpoint found - run the training cell first.")
        return

    eval_model = UNetModel(in_channels=3, out_channels=1).to(device)
    ckpt_data = torch.load(ckpt, map_location=device, weights_only=False)
    eval_model.load_state_dict(ckpt_data["net_state_dict"])
    eval_model.eval()
    print(f"Loaded checkpoint (epoch {ckpt_data.get('epoch', '?')}) from {ckpt}")

    test_ds = get_loader(dataset_type="test", batch_size=1, shuffle=False)
    out_dir = "/content/mar20_results"
    os.makedirs(out_dir, exist_ok=True)

    with torch.no_grad():
        for i, (data, target) in enumerate(tqdm(test_ds, desc="Inference")):
            pred_np = torch.sigmoid(eval_model(data.to(device))).squeeze().cpu().numpy()
            target_np = target.squeeze().numpy()

            fig, axes = plt.subplots(1, 2, figsize=(12, 5))
            im1 = axes[0].imshow(pred_np, cmap="viridis")
            axes[0].set_title("Prediction")
            plt.colorbar(im1, ax=axes[0])
            im2 = axes[1].imshow(target_np, cmap="viridis")
            axes[1].set_title("Ground Truth")
            plt.colorbar(im2, ax=axes[1])
            plt.tight_layout()
            plt.savefig(os.path.join(out_dir, f"result_{i:04d}.png"), dpi=100)
            plt.close(fig)

            if i + 1 >= num_samples:
                break

    print(f"Saved {min(num_samples, i + 1)} result images to {out_dir}/")


visualize_results(num_samples=100)

## 7. Save & download results

Colab VMs are wiped when the session ends, so copy the checkpoints/results to
Google Drive (`DRIVE_SAVE_DIR`) and/or download them as a zip.

In [ ]:
import os
import shutil
import zipfile


def to_zip(src_dir, dst_zip):
    with zipfile.ZipFile(dst_zip, "w", zipfile.ZIP_DEFLATED) as z:
        for root, _, files in os.walk(src_dir):
            for name in files:
                full = os.path.join(root, name)
                rel = os.path.relpath(full, start=os.path.dirname(src_dir))
                z.write(full, rel)


DRIVE_SAVE_DIR = "/content/drive/MyDrive/dl4rs_MAR20/checkpoints"  # change as needed

# 1) Local zip bundle (always available)
bundle = "/content/mar20_checkpoints.zip"
to_zip(CKPT_DIR, bundle)
print("Bundle created:", bundle, f"({os.path.getsize(bundle) / 1e6:.0f} MB)")

if os.path.isdir("/content/mar20_results"):
    to_zip("/content/mar20_results", "/content/mar20_results.zip")
    print("Results zip created: /content/mar20_results.zip")

# 2) Copy to Google Drive (only if you mounted it above)
if os.path.isdir("/content/drive/MyDrive"):
    os.makedirs(DRIVE_SAVE_DIR, exist_ok=True)
    for entry in os.listdir(CKPT_DIR):
        src = os.path.join(CKPT_DIR, entry)
        dst = os.path.join(DRIVE_SAVE_DIR, entry)
        if os.path.isfile(src):
            shutil.copy2(src, dst)
        else:
            shutil.copytree(src, dst, dirs_exist_ok=True)
    print("Checkpoints copied to", DRIVE_SAVE_DIR)
else:
    print("Drive not mounted - skipping Drive copy (run the Drive mount cell).")

# 3) Trigger the download dialog
from google.colab import files
files.download(bundle)

---

### Notes / known limits
- **Memory**: the loader preloads a whole split into RAM (~4.6 GB for `train`).
  The training cell only loads `train` + `val`, which fits comfortably in a
  standard Colab session.
- **Reproducibility**: results differ slightly run-to-run (CUDA non-determinism);
  a global seed is set in the training cell.
- **Checkpoint format** is identical to the local repo, so
  `result_script.py` or the local `result_script`/`tests.py` can load
  `best_model.pt` without changes.